# S6E8 — modelling pipeline

The experiment notebook. `eda-s6e8.ipynb` is the frozen EDA record; this is where probes run.

Everything that varies between runs lives in one `CFG` dict, overridable from the `S6E8_CFG`
environment variable. On Kaggle the variable is absent and the defaults below are used, so the same
file is both the local probe runner and the pushed kernel.

**Strict-twin discipline** (`KAGGLE_PLAYBOOK.md` §3): a probe changes exactly one `CFG` field from
the champion. `scripts/run_local.py --diff-vs` checks this before the run starts.

In [ ]:
import json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
T_START = time.time()

# ---- frozen contract (README.md). Never overridable by CFG. ----------------
SEED, N_FOLDS = 42, 5
TARGET, ID = "addicted_label", "id"
COMP = "playground-series-s6e8"

# ---- everything a probe is allowed to change ------------------------------
DEFAULTS = {
    "run_tag":       "champion",
    "learner":       "lgb",        # lgb | xgb | cat | mlp | realmlp | ftt | tabtf
    "model_seed":    42,           # model randomness ONLY -- never the CV split seed
    "fe_interaction": False,       # B1: opens_per_hour, notif_per_hour, notif_per_open
    "fe_composition": False,       # B2: other_hours, share_*, weekend_ratio, screen_total
    "fe_normalization": False,     # B3: free_time, screen_per_sleep, screen_per_age, social_per_age
    "drop_flat_cats": False,       # B5: drop gender/stress_level/academic_work_impact
    "num_as_cat":    False,        # D2: declare the 9 numeric columns CATEGORICAL, acting on the
                                   # EDA finding that their value->target map is a lookup table
                                   # (rates 0.119..0.986, Pearson(value, rate) = -0.044) rather
                                   # than a function -- i.e. the values are labels, not magnitudes
    "te_cols":       [],           # B6: per-value TARGET encoding of these columns ("num" = all numeric)
    "te_crosses":    [],           # B7: target-encode VALUE PAIRS, e.g. [["a","b"]]
    "te_smooth":     20.0,         # prior weight in the smoothed rate
    "te_inner_folds": 5,           # inner OOF folds used to encode the training rows
    "mlp_embed":     [],           # E2: learn an nn.Embedding per listed column instead of
                                   # feeding its value. The regularised answer to D2: a split
                                   # on a 231-level column overfits, an embedding under weight
                                   # decay is the same lookup fitted with a penalty.
    "lgb_params":    {},           # merged over the baseline params (all learners read it)
    "n_estimators":  8000,
    "early_stop":    400,
}
KAGGLE_DIR = Path("/kaggle/input/competitions") / COMP
ON_KAGGLE  = KAGGLE_DIR.exists()

# Pinned config for KERNEL runs. Kaggle offers no way to pass an env var into a
# kernel, so a kernel run is configured by editing this dict before
# `kaggle kernels push`. Ignored locally, where run_local.py sets S6E8_CFG instead.
# Kernels exist for runs too long for the ~10-minute local budget (Kaggle allows 9h).
KAGGLE_CFG = {"run_tag": "D1_catboost", "learner": "cat", "te_cols": "num"}

CFG = {**DEFAULTS,
       **(KAGGLE_CFG if ON_KAGGLE else {}),
       **json.loads(os.environ.get("S6E8_CFG", "{}"))}
assert set(CFG) <= set(DEFAULTS), f"unknown CFG keys: {set(CFG) - set(DEFAULTS)}"
DATA_DIR   = KAGGLE_DIR if ON_KAGGLE else Path("data")
OUT_DIR    = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("S6E8_OUT", "experiments/preds/local"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"on_kaggle={ON_KAGGLE}  out={OUT_DIR}")
print("CFG:", json.dumps(CFG, indent=1))

## Load and feature engineering

**Leakage note.** Every engineered feature below is *row-wise arithmetic on that row's own values* —
no aggregation across rows, no reference to `y`. It is therefore trivially leak-free and correctly
computed once, before the CV loop. This is the deliberate exception to `README.md`'s
fit-inside-the-fold rule: that rule exists for transforms that *learn* something from the data
(target encoders, bin edges, scalers), and none of these do. Do not "fix" this by moving it into the
fold loop — it would change nothing and cost 5× the compute.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
sub   = pd.read_csv(DATA_DIR / "sample_submission.csv")

RAW_CAT = ["gender", "stress_level", "academic_work_impact"]
RAW_NUM = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
           "work_study_hours", "sleep_hours", "notifications_per_day",
           "app_opens_per_day", "weekend_screen_time"]

# Denominators must be strictly positive or the ratios below silently produce inf.
# Assert it rather than trusting the EDA snapshot.
for col, lo in [("daily_screen_time_hours", 0), ("app_opens_per_day", 0),
                ("sleep_hours", 0), ("age", 0)]:
    for name, df in (("train", train), ("test", test)):
        bad = (df[col].dropna() <= lo).sum()
        assert bad == 0, f"{name}.{col} has {bad} values <= {lo}; ratios would be inf"
print("denominator safety: all strictly positive in train and test")


def engineer(df):
    # Row-wise, unsupervised, leak-free by construction. Returns (df, added_cols).
    out, added = df.copy(), []

    if CFG["fe_interaction"]:
        # B1. The rank-gap finding: these two rank 9th/6th on marginal AUC but 1st/2nd on
        # split usage. A GBDT cannot express x/y with axis-aligned splits, so it burns
        # capacity approximating these; hand them over explicitly.
        out["opens_per_hour"] = df.app_opens_per_day / df.daily_screen_time_hours
        out["notif_per_hour"] = df.notifications_per_day / df.daily_screen_time_hours
        out["notif_per_open"] = df.notifications_per_day / df.app_opens_per_day
        added += ["opens_per_hour", "notif_per_hour", "notif_per_open"]

    if CFG["fe_composition"]:
        # B2. daily_screen_time is a confirmed composition: total - parts is NEVER negative
        # across 374,297 complete rows, so "other_hours" is a real latent variable.
        parts = df.social_media_hours + df.gaming_hours + df.work_study_hours
        out["other_hours"]   = df.daily_screen_time_hours - parts
        out["share_social"]  = df.social_media_hours / df.daily_screen_time_hours
        out["share_gaming"]  = df.gaming_hours / df.daily_screen_time_hours
        out["share_work"]    = df.work_study_hours / df.daily_screen_time_hours
        out["share_other"]   = out["other_hours"] / df.daily_screen_time_hours
        out["weekend_ratio"] = df.weekend_screen_time / df.daily_screen_time_hours
        out["screen_total"]  = df.daily_screen_time_hours + df.weekend_screen_time
        added += ["other_hours", "share_social", "share_gaming", "share_work",
                  "share_other", "weekend_ratio", "screen_total"]

    if CFG["fe_normalization"]:
        # B3. Absolute screen time means something different at 18 than at 35, and
        # against 5 vs 9 hours of sleep.
        out["free_time"]        = 24 - df.sleep_hours - df.daily_screen_time_hours
        out["screen_per_sleep"] = df.daily_screen_time_hours / df.sleep_hours
        out["screen_per_age"]   = df.daily_screen_time_hours / df.age
        out["social_per_age"]   = df.social_media_hours / df.age
        added += ["free_time", "screen_per_sleep", "screen_per_age", "social_per_age"]

    return out, added


train_fe, added = engineer(train)
test_fe,  _     = engineer(test)

cat_cols = [] if CFG["drop_flat_cats"] else RAW_CAT
FEATURES = RAW_NUM + added + cat_cols
print(f"{len(FEATURES)} features = {len(RAW_NUM)} raw num + {len(added)} engineered + {len(cat_cols)} cat")
print(f"engineered: {added or '(none)'}")

In [ ]:
# ---- FE correctness, asserted not assumed ---------------------------------
for name, df in (("train", train_fe), ("test", test_fe)):
    for c in added:
        v = df[c].to_numpy(dtype="float64")
        finite = v[~np.isnan(v)]
        assert np.isfinite(finite).all(), f"{name}.{c} contains inf"
if CFG["fe_composition"]:
    for name, df in (("train", train_fe), ("test", test_fe)):
        neg = (df["other_hours"].dropna() < -1e-9).sum()
        assert neg == 0, f"{name}.other_hours negative on {neg} rows -- composition assumption broken"
    print("composition check: other_hours >= 0 everywhere in train and test")

# Row-wise purity: recomputing FE on a shuffled subset must give identical values.
# This is what proves no cross-row aggregation crept in (which WOULD be a leak).
if added:
    probe = train.sample(2000, random_state=0)
    re_fe, _ = engineer(probe)
    ref = train_fe.loc[probe.index, added]
    assert np.allclose(re_fe[added].to_numpy(dtype="float64"),
                       ref.to_numpy(dtype="float64"), equal_nan=True), \
        "FE is not row-wise -- values changed when computed on a subset"
    print("row-wise purity: FE identical on a 2000-row shuffled subset -> no cross-row leakage")

print(f"\nengineered-column coverage (non-null %):")
for c in added:
    print(f"  {c:<20} {train_fe[c].notna().mean()*100:5.1f}%")

## Per-value target encoding (supervised — leakage discipline applies here)

Everything above is unsupervised row arithmetic. **This is not.** It maps each raw *value* to the
rate at which that value's rows are positive, so it touches `y` and must be fit strictly inside the
training fold — `README.md`'s load-bearing rule.

Why it should work here, and why B1 could not: the value→target relationship is **non-monotone**.
Individual `notifications_per_day` values carry positive rates spanning **0.119 to 0.986**, yet
Pearson(value, rate) is −0.044 and only 47.8% of consecutive steps increase. AUC only sees monotone
association, which is why the raw column scores 0.492; a tree *can* carve out individual values,
which is why it spent the most splits there. No unsupervised transform — ratio, sum, or quantile
bin — can reconstruct a lookup table. Playbook §6: *ask whether the lever is supervised before
building the unsupervised imitation of it.*

**Nesting.** A single fit on the outer training fold would let each training row see its own label
through its value's mean, so the model would learn to trust the encoding far more than it should and
the OOF would be inflated. Instead:

- training rows are encoded **out-of-fold within the training fold** (inner K-fold);
- validation and test rows are encoded with the mapping fit on the **whole** outer training fold;
- unseen values fall back to the training fold's prior, never to a global statistic and never raising.

In [ ]:
TE_COLS = RAW_NUM if CFG["te_cols"] == "num" else list(CFG["te_cols"])
TE_COLS = [c for c in TE_COLS if c in FEATURES]

# Integer codes per distinct value, computed ONCE over train+test so the code space
# is shared. This is an unsupervised label mapping (it never touches y), so doing it
# outside the fold loop is not a leak -- same argument as the categorical vocabulary.
# NaN gets its own code rather than being dropped, so "missing" carries its own rate.
TE_CODES = {}
for c in TE_COLS:
    codes, _ = pd.factorize(pd.concat([train_fe[c], test_fe[c]]), use_na_sentinel=False)
    TE_CODES[c] = (codes[:len(train_fe)], codes[len(train_fe):], int(codes.max()) + 1)

def _rate_table(codes_fit, y_fit, k, prior, m):
    # Smoothed per-value positive rate via bincount -- O(n), no groupby/map.
    s = np.bincount(codes_fit, weights=y_fit, minlength=k)
    n = np.bincount(codes_fit, minlength=k)
    return (s + prior * m) / (n + m)          # unseen value -> n=0 -> exactly the prior

def target_encode(c, itr, iva, seed, n_inner, m):
    # Returns (enc_train, enc_val, enc_test) for column c.
    # y is read ONLY at rows in itr -- the outer training fold.
    ctr_all, cte_all, k = TE_CODES[c]
    y_tr = y[itr]
    prior = float(y_tr.mean())

    enc_tr = np.empty(len(itr))
    inner = StratifiedKFold(n_splits=n_inner, shuffle=True, random_state=seed)
    for i_fit, i_app in inner.split(np.zeros(len(itr)), y_tr):
        tbl = _rate_table(ctr_all[itr][i_fit], y_tr[i_fit], k, prior, m)
        enc_tr[i_app] = tbl[ctr_all[itr][i_app]]

    full = _rate_table(ctr_all[itr], y_tr, k, prior, m)
    return enc_tr, full[ctr_all[iva]], full[cte_all]

# B7: value CROSSES. The supervised version of what B1 tried and failed at -- if the
# single-column relationship is a lookup table, the pairwise one may be too, and a
# cross is the only encoding that can express "this combination of values" rather than
# "this value of x, adjusted for y".
#
# Cardinality is the whole risk here. A cross of two columns with 231 and 166 levels
# has up to 38k cells over 691k rows (~18/cell), versus ~2700 rows/value for a single
# column. Self-influence per row rises from ~0.03% to ~3%, so the nesting and the
# smoothing stop being a formality -- and the train/val gap audit, which had no power
# on single columns, becomes a real test.
for a, b in CFG["te_crosses"]:
    ca_tr, ca_te, ka = TE_CODES[a] if a in TE_CODES else (None, None, None)
    if ca_tr is None:
        codes, _ = pd.factorize(pd.concat([train_fe[a], test_fe[a]]), use_na_sentinel=False)
        ca_tr, ca_te, ka = codes[:len(train_fe)], codes[len(train_fe):], int(codes.max()) + 1
    cb_tr, cb_te, kb = TE_CODES[b] if b in TE_CODES else (None, None, None)
    if cb_tr is None:
        codes, _ = pd.factorize(pd.concat([train_fe[b], test_fe[b]]), use_na_sentinel=False)
        cb_tr, cb_te, kb = codes[:len(train_fe)], codes[len(train_fe):], int(codes.max()) + 1
    name = f"{a}_X_{b}"
    joint = np.concatenate([ca_tr * kb + cb_tr, ca_te * kb + cb_te])
    dense, _ = pd.factorize(joint)                      # compact the sparse product space
    TE_CODES[name] = (dense[:len(train_fe)], dense[len(train_fe):], int(dense.max()) + 1)
    TE_COLS.append(name)
    occ = np.bincount(dense[:len(train_fe)])
    print(f"cross {name}: {TE_CODES[name][2]:,} observed cells, "
          f"{len(train_fe)/TE_CODES[name][2]:.1f} rows/cell median-occupancy {int(np.median(occ))}")

print(f"target-encoding {len(TE_COLS)} column(s): {TE_COLS or '(none)'}")
if TE_COLS:
    print(f"  smoothing m={CFG['te_smooth']}, inner folds={CFG['te_inner_folds']}", flush=True)

## Cross-validation

The frozen split, unchanged: `StratifiedKFold(5, shuffle=True, random_state=42)`. `CFG["model_seed"]`
moves the *model's* randomness only — the partition never moves, which is what keeps every archived
OOF matrix blendable with every other.

In [ ]:
X, X_test = train_fe[FEATURES].copy(), test_fe[FEATURES].copy()

# D2: the numerics can also be declared categorical. The vocabulary is still built over
# train + test and never touches y, so this stays an UNSUPERVISED label mapping -- the
# same argument that licenses hoisting it out of the CV loop for the real categoricals.
# True = all nine; a list = just those columns. D2 showed the distinction is the whole
# point: the screen-time columns are strongly MONOTONE (solo AUC 0.86-0.89 over 700-1400
# levels), so categorifying them throws away real ordinal signal and hands LightGBM a
# 1400-level feature to overfit -- -0.004 OOF. The lookup-table structure was only ever
# a property of the low-AUC, high-split-usage columns.
NUM_AS_CAT  = (RAW_NUM if CFG["num_as_cat"] is True
               else [c for c in (CFG["num_as_cat"] or []) if c in FEATURES])
AS_CATEGORY = cat_cols + NUM_AS_CAT
for c in AS_CATEGORY:
    levels = pd.Index(sorted(set(train[c].dropna()) | set(test[c].dropna())))
    X[c]      = pd.Categorical(X[c], categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)
if NUM_AS_CAT:
    print("num_as_cat: " + ", ".join(f"{c}={X[c].cat.categories.size} levels" for c in NUM_AS_CAT))
y = train[TARGET].values

BASE_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,
                   n_jobs=-1, verbose=-1)
params = {**BASE_PARAMS, **CFG["lgb_params"],
          "n_estimators": CFG["n_estimators"], "random_state": CFG["model_seed"]}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
assert skf.n_splits == 5 and skf.shuffle is True and skf.random_state == 42, "CV split drifted!"

oof        = np.zeros(len(X))
fold_id    = np.full(len(X), -1, dtype=int)
test_sum   = np.zeros(len(X_test), dtype=np.float64)
fold_aucs, best_iters = [], []

for k, (itr, iva) in enumerate(skf.split(X, y)):
    fold_id[iva] = k
    Xtr, Xva, Xte = X.iloc[itr].copy(), X.iloc[iva].copy(), X_test.copy()

    # Supervised encoding -- re-fit from scratch inside THIS fold every time.
    # Never hoist this out of the loop: it is the one transform here that sees y.
    for c in TE_COLS:
        e_tr, e_va, e_te = target_encode(
            c, itr, iva, seed=SEED + k,
            n_inner=CFG["te_inner_folds"], m=CFG["te_smooth"])
        Xtr[f"{c}__te"], Xva[f"{c}__te"], Xte[f"{c}__te"] = e_tr, e_va, e_te

    if CFG["learner"] == "lgb":
        model = lgb.LGBMClassifier(**params)
        model.fit(Xtr, y[itr], eval_set=[(Xva, y[iva])], eval_metric="auc",
                  callbacks=[lgb.early_stopping(CFG["early_stop"], verbose=False)])
        best_it = model.best_iteration_
    elif CFG["learner"] == "xgb":
        import xgboost as xgb
        # lgb_params is the single knob dict for every learner, so XGB-specific keys
        # arrive through it and must OVERRIDE these defaults rather than collide with
        # them -- passing max_depth both ways is a duplicate-kwarg TypeError.
        xgb_kw = dict(
            n_estimators=CFG["n_estimators"], learning_rate=params["learning_rate"],
            max_depth=6, tree_method="hist", enable_categorical=True,
            eval_metric="auc", early_stopping_rounds=CFG["early_stop"],
            random_state=CFG["model_seed"], n_jobs=-1)
        xgb_kw.update({k: v for k, v in CFG["lgb_params"].items()
                       if k not in ("num_leaves", "min_child_samples")})
        model = xgb.XGBClassifier(**xgb_kw)
        model.fit(Xtr, y[itr], eval_set=[(Xva, y[iva])], verbose=False)
        best_it = model.best_iteration
    elif CFG["learner"] == "cat":
        from catboost import CatBoostClassifier
        # CatBoost rejects NaN inside a categorical feature, so missing gets its own
        # explicit level. That is an UNSUPERVISED relabelling (it never touches y) --
        # same argument as the pandas category vocabulary, not a leak.
        def _cat_ready(df):
            d = df.copy()
            for c in AS_CATEGORY:
                d[c] = d[c].astype(object).where(d[c].notna(), "__NA__").astype(str)
            return d
        Ctr, Cva, Cte = _cat_ready(Xtr), _cat_ready(Xva), _cat_ready(Xte)
        model = CatBoostClassifier(
            iterations=CFG["n_estimators"], learning_rate=params["learning_rate"],
            depth=6, eval_metric="AUC", random_seed=CFG["model_seed"],
            early_stopping_rounds=CFG["early_stop"], thread_count=-1,
            allow_writing_files=False, verbose=False,
            **{k: v for k, v in CFG["lgb_params"].items()
               if k not in ("learning_rate", "num_leaves", "min_child_samples")})
        model.fit(Ctr, y[itr], eval_set=(Cva, y[iva]), cat_features=AS_CATEGORY, verbose=False)
        best_it = model.get_best_iteration()   # read-only property, unlike lgb/xgb
        Xva, Xte = Cva, Cte      # predict from the same representation we trained on
    elif CFG["learner"] == "mlp":
        # A FOURTH FAMILY. Every within-family tuning probe has missed the gate (C1's
        # blend, C2, C3, D2, D2b); the only win since target encoding came from adding a
        # family (D1 CatBoost). A neural net shares no inductive bias with a tree at all,
        # so it is the largest remaining decorrelation bet.
        import torch, torch.nn as nn
        hp = {"hidden": [256, 128, 64], "dropout": 0.15, "lr": 1e-3, "wd": 1e-4,
              "batch": 4096, "epochs": 60, **CFG["lgb_params"]}
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        torch.manual_seed(CFG["model_seed"])

        # E2 embeddings. Codes come from TE_CODES, which was factorised over train + test and
        # never touches y -- unsupervised, so it is safe outside the fold loop. The embedded
        # column is ALSO left in the numeric block, exactly as B1 kept its raw columns.
        emb_cols = [c for c in CFG["mlp_embed"] if c in TE_CODES]
        missing_emb = [c for c in CFG["mlp_embed"] if c not in TE_CODES]
        assert not missing_emb, f"mlp_embed needs these in te_cols to get codes: {missing_emb}"
        emb_spec = [(c, TE_CODES[c][2], int(min(24, round(TE_CODES[c][2] ** 0.5) + 1)))
                    for c in emb_cols]
        if emb_spec:
            print("    embeddings: " + ", ".join(f"{c}({k}->{d})" for c, k, d in emb_spec))

        def codes_for(which):
            if not emb_cols:
                return None
            cols = []
            for c in emb_cols:
                ctr, cte, _ = TE_CODES[c]
                cols.append({"tr": ctr[itr], "va": ctr[iva], "te": cte}[which])
            return torch.from_numpy(np.stack(cols, 1).astype("int64"))

        num_cols = [c for c in Xtr.columns if c not in cat_cols]

        # Imputation and standardisation are fit on the TRAINING FOLD ONLY and applied to
        # val/test -- the same rule as the target encoder. A tree needed neither, so this
        # is the first place in the notebook where scaling discipline is load-bearing.
        med = Xtr[num_cols].median()
        mu  = Xtr[num_cols].fillna(med).mean()
        sd  = Xtr[num_cols].fillna(med).std().replace(0, 1.0)

        def to_tensor(df):
            num = ((df[num_cols].fillna(med) - mu) / sd).to_numpy(dtype="float32")
            miss = df[num_cols].isna().to_numpy(dtype="float32")       # missingness is signal
            parts = [num, miss]
            for c in cat_cols:
                codes = df[c].cat.codes.to_numpy()                     # -1 for NaN
                oh = np.zeros((len(df), len(df[c].cat.categories) + 1), dtype="float32")
                oh[np.arange(len(df)), codes + 1] = 1.0                # column 0 == missing
                parts.append(oh)
            return torch.from_numpy(np.hstack(parts))

        Ttr, Tva, Tte = to_tensor(Xtr), to_tensor(Xva), to_tensor(Xte)
        Ctr, Cva, Cte = codes_for("tr"), codes_for("va"), codes_for("te")
        ytr_t = torch.from_numpy(y[itr].astype("float32"))

        class Net(nn.Module):
            def __init__(self, d_num):
                super().__init__()
                self.emb = nn.ModuleList([nn.Embedding(k, d) for _, k, d in emb_spec])
                d_in = d_num + sum(d for _, _, d in emb_spec)
                blocks = []
                for h in hp["hidden"]:
                    blocks += [nn.Linear(d_in, h), nn.BatchNorm1d(h), nn.GELU(),
                               nn.Dropout(hp["dropout"])]
                    d_in = h
                blocks += [nn.Linear(d_in, 1)]
                self.body = nn.Sequential(*blocks)

            def forward(self, xn, xc=None):
                if self.emb:
                    xn = torch.cat([xn] + [e(xc[:, i]) for i, e in enumerate(self.emb)], 1)
                return self.body(xn)

        net = Net(Ttr.shape[1]).to(dev)
        opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=hp["epochs"])
        lossf = nn.BCEWithLogitsLoss()

        def infer(T, C=None):
            net.eval(); out = []
            with torch.no_grad():
                for i in range(0, len(T), 16384):
                    cb = None if C is None else C[i:i+16384].to(dev)
                    out.append(torch.sigmoid(net(T[i:i+16384].to(dev), cb).squeeze(1)).cpu().numpy())
            return np.concatenate(out)

        # Early stopping on ROC AUC -- the competition metric, never a loss proxy (README).
        best_auc, best_state, best_ep, since = -1.0, None, 0, 0
        n = len(Ttr)
        for ep in range(hp["epochs"]):
            net.train()
            perm = torch.randperm(n)
            for i in range(0, n, hp["batch"]):
                idx = perm[i:i+hp["batch"]]
                opt.zero_grad()
                cb = None if Ctr is None else Ctr[idx].to(dev)
                loss = lossf(net(Ttr[idx].to(dev), cb).squeeze(1), ytr_t[idx].to(dev))
                loss.backward(); opt.step()
            sched.step()
            a = roc_auc_score(y[iva], infer(Tva, Cva))
            if a > best_auc:
                best_auc, best_ep, since = a, ep, 0
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            else:
                since += 1
                if since >= 8:
                    break
        net.load_state_dict(best_state)

        class _Wrap:
            # Minimal predict_proba shim so the shared fold code below is untouched.
            def __init__(self, va, te): self.va, self.te = va, te
            def predict_proba(self, X_):
                p = self.va if len(X_) == len(Xva) else self.te
                return np.column_stack([1 - p, p])
        model = _Wrap(infer(Tva, Cva), infer(Tte, Cte))
        best_it = best_ep
        n_emb = sum(d for _, _, d in emb_spec)
        print(f"    mlp: {Ttr.shape[1]}+{n_emb} in, best epoch {best_ep} "
              f"(val AUC {best_auc:.6f}), dev={dev}", flush=True)
    elif CFG["learner"] == "realmlp":
        # RealMLP (pytabkit) -- the strongest published tabular-NN baseline, and NOT a
        # reshaped version of our own MLP: it brings periodic-linear (PLR) numeric
        # embeddings, a bespoke scaling/tfms pipeline and its own LR schedule. Its own
        # validation split is passed in explicitly so it never sees the val fold's labels
        # through an internal random split.
        from pytabkit import RealMLP_TD_Classifier
        hp = {"n_epochs": 60, "batch_size": 1024, **CFG["lgb_params"]}
        Rtr, Rva, Rte = Xtr.copy(), Xva.copy(), Xte.copy()
        for c in cat_cols:                       # pytabkit wants plain strings, no NaN
            for d in (Rtr, Rva, Rte):
                d[c] = d[c].astype(object).where(d[c].notna(), "__NA__").astype(str)
        # RealMLP rejects NaN in continuous columns. Impute with the TRAINING FOLD median
        # and keep an explicit missing indicator -- the EDA measured a non-trivial AUC on
        # several missing-indicators, so dropping that signal would be a silent loss.
        r_num = [c for c in Xtr.columns if c not in cat_cols]
        r_med = Rtr[r_num].median()
        for d in (Rtr, Rva, Rte):
            for c in r_num:
                if Xtr[c].isna().any() or d[c].isna().any():
                    d[f"{c}__isna"] = d[c].isna().astype("float32")
            d[r_num] = d[r_num].fillna(r_med)
        model = RealMLP_TD_Classifier(
            device="cuda" if __import__("torch").cuda.is_available() else "cpu",
            random_state=CFG["model_seed"], verbosity=0,
            val_metric_name="1-auc_ovr",   # binary auc_ovr IS roc_auc -- the metric, not a proxy
            **hp)
        model.fit(Rtr, y[itr], X_val=Rva, y_val=y[iva], cat_col_names=cat_cols)
        Xva, Xte = Rva, Rte
        best_it = int(hp["n_epochs"])

    elif CFG["learner"] in ("ftt", "tabtf"):
        # FT-Transformer / TabTransformer. Attention over feature tokens is a genuinely
        # different inductive bias from both an MLP and a tree: it models feature-to-feature
        # interaction explicitly rather than through partitioning or a dense mixture.
        import torch, torch.nn as nn
        hp = {"d_token": 64, "n_blocks": 3, "heads": 8, "dropout": 0.1, "lr": 1e-3,
              "wd": 1e-5, "batch": 4096, "epochs": 30, "patience": 5, **CFG["lgb_params"]}
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        torch.manual_seed(CFG["model_seed"])
        num_cols = [c for c in Xtr.columns if c not in cat_cols]
        card = [len(Xtr[c].cat.categories) + 1 for c in cat_cols]

        med = Xtr[num_cols].median()
        mu  = Xtr[num_cols].fillna(med).mean()
        sd  = Xtr[num_cols].fillna(med).std().replace(0, 1.0)

        def tens(df):
            n = torch.from_numpy(((df[num_cols].fillna(med) - mu) / sd)
                                 .to_numpy(dtype="float32"))
            c = torch.from_numpy(np.stack(
                [df[col].cat.codes.to_numpy() + 1 for col in cat_cols], 1).astype("int64"))
            return n, c

        (Ntr, Ktr), (Nva, Kva), (Nte, Kte) = tens(Xtr), tens(Xva), tens(Xte)
        ytr_t = torch.from_numpy(y[itr].astype("float32"))

        if CFG["learner"] == "ftt":
            from rtdl_revisiting_models import FTTransformer
            net = FTTransformer(n_cont_features=len(num_cols), cat_cardinalities=card,
                                d_out=1, n_blocks=hp["n_blocks"], d_block=hp["d_token"],
                                attention_n_heads=hp["heads"],
                                attention_dropout=hp["dropout"], ffn_d_hidden=None,
                                ffn_d_hidden_multiplier=4 / 3,
                                ffn_dropout=hp["dropout"], residual_dropout=0.0).to(dev)
            fwd = lambda n_, c_: net(n_, c_).squeeze(-1)
        else:
            from tab_transformer_pytorch import TabTransformer
            net = TabTransformer(categories=tuple(card), num_continuous=len(num_cols),
                                 dim=hp["d_token"], depth=hp["n_blocks"], heads=hp["heads"],
                                 dim_out=1, attn_dropout=hp["dropout"],
                                 ff_dropout=hp["dropout"],
                                 mlp_hidden_mults=(4, 2),
                                 mlp_act=nn.ReLU()).to(dev)   # default None lands in a
                                 # Sequential and raises 'NoneType' is not callable
            fwd = lambda n_, c_: net(c_, n_).squeeze(-1)

        opt = torch.optim.AdamW(net.parameters(), lr=hp["lr"], weight_decay=hp["wd"])
        lossf = nn.BCEWithLogitsLoss()

        def infer(N, K):
            net.eval(); out = []
            with torch.no_grad():
                for i in range(0, len(N), 8192):
                    out.append(torch.sigmoid(
                        fwd(N[i:i+8192].to(dev), K[i:i+8192].to(dev))).cpu().numpy())
            return np.concatenate(out)

        best_auc, best_state, best_it, since = -1.0, None, 0, 0
        for ep in range(hp["epochs"]):
            net.train()
            perm = torch.randperm(len(Ntr))
            for i in range(0, len(Ntr), hp["batch"]):
                idx = perm[i:i+hp["batch"]]
                opt.zero_grad()
                loss = lossf(fwd(Ntr[idx].to(dev), Ktr[idx].to(dev)), ytr_t[idx].to(dev))
                loss.backward(); opt.step()
            a = roc_auc_score(y[iva], infer(Nva, Kva))          # AUC, never a loss proxy
            print(f"      ep {ep:2d} val AUC {a:.6f}", flush=True)
            if a > best_auc:
                best_auc, best_it, since = a, ep, 0
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
            else:
                since += 1
                if since >= hp["patience"]:
                    break
        net.load_state_dict(best_state)

        class _WrapT:
            def __init__(self, va, te): self.va, self.te = va, te
            def predict_proba(self, X_):
                p = self.va if len(X_) == len(Xva) else self.te
                return np.column_stack([1 - p, p])
        model = _WrapT(infer(Nva, Kva), infer(Nte, Kte))
    else:
        raise ValueError(f"unknown learner {CFG['learner']}")
    oof[iva] = model.predict_proba(Xva)[:, 1]
    # Accumulate the SUM in float64 and divide once, rather than adding p/N_FOLDS each
    # fold. XGBoost returns float32 and does emit exactly 1.0; five rounded p/5 terms
    # then sum to 1.0000000149, which the submission's between(0,1) check rejects.
    test_sum += model.predict_proba(Xte)[:, 1].astype(np.float64)
    fold_aucs.append(roc_auc_score(y[iva], oof[iva]))
    best_iters.append(int(best_it))
    print(f"  fold {k}: AUC={fold_aucs[-1]:.6f}  best_iter={best_iters[-1]}", flush=True)

test_proba = np.clip(test_sum / N_FOLDS, 0.0, 1.0)   # clip is rank-preserving, so AUC is unaffected
oof_auc = roc_auc_score(y, oof)
assert (fold_id >= 0).all()
assert max(best_iters) < CFG["n_estimators"], \
    f"a fold hit the n_estimators cap ({max(best_iters)}) -- under-trained, raise it"

print(f"\nOOF AUC : {oof_auc:.6f}")
print(f"folds   : {np.mean(fold_aucs):.6f} +/- {np.std(fold_aucs):.6f}  (spread {max(fold_aucs)-min(fold_aucs):.6f})")
print("fold spread is EVALUATION-FOLD DIFFICULTY, not test-prediction variance.")

In [ ]:
if hasattr(model, "feature_importances_"):      # trees only; the MLP wrapper has none
    imp = (pd.Series(model.feature_importances_, index=list(Xtr.columns))
           / model.feature_importances_.sum() * 100).sort_values(ascending=False)
    print("feature importance (last fold, % of splits):")
    print(imp.round(2).to_string())
else:
    print(f"{CFG['learner']}: no split-based feature importance")

In [ ]:
# ---- leakage audit for the supervised encoder -----------------------------
# Reported honestly, because a train/val AUC gap is a WEAK test on this data and it
# would be worse than useless to treat a small gap as proof of safety.
#
# A row's own label moves its value's mean by 1/(n_rows_for_that_value + m). Columns
# like notifications_per_day have ~2700 rows per value, so self-influence is ~0.03%
# and even a naive (un-nested) fit shows almost no gap -- measured directly: a
# deliberately leaky fit produced a gap of just +0.0008. So a small gap here is
# evidence of low cardinality, NOT evidence the nesting works.
#
# What actually provides safety, in order of importance:
#   1. min rows-per-value (printed below) -- the structural bound on self-influence;
#   2. the inner-OOF nesting, which matters precisely for the singleton values that
#      columns like daily_screen_time_hours DO have (min n = 1);
#   3. smoothing, which shrinks a singleton toward the prior.
if TE_COLS:
    print(f"{'column':<42}{'med n/val':>10}{'AUC tr':>9}{'AUC va':>9}{'gap':>9}")
    for c in TE_COLS:
        occ = np.bincount(TE_CODES[c][0][itr])
        med_n = int(np.median(occ[occ > 0]))
        a_tr = roc_auc_score(y[itr], Xtr[f"{c}__te"])
        a_va = roc_auc_score(y[iva], Xva[f"{c}__te"])
        gap = a_tr - a_va
        flag = ""
        if med_n <= 30:
            flag = "  <-- sparse cells: gap test HAS power here"
        if gap > 0.02:
            flag = "  <-- LEAK SUSPECT"
        print(f"{c:<42}{med_n:>10}{a_tr:>9.4f}{a_va:>9.4f}{gap:>+9.4f}{flag}")
    print("\nInterpret the gap only as a smoke alarm for a GROSS error (e.g. encoding fit")
    print("on the full frame including val). It cannot certify the nesting; the nesting is")
    print("certified by construction -- the inner split never shows a row its own label.")

## Artifacts

In [ ]:
LEARNER = CFG["learner"]
pd.DataFrame({ID: train[ID], "fold": fold_id, "proba": oof}).to_csv(
    OUT_DIR / f"oof_proba_{LEARNER}.csv", index=False)
pd.DataFrame({ID: test[ID], "proba": test_proba}).to_csv(
    OUT_DIR / f"test_proba_{LEARNER}.csv", index=False)

submission = pd.DataFrame({ID: test[ID], TARGET: test_proba})
submission.to_csv(OUT_DIR / "submission.csv", index=False)

assert list(submission.columns) == [ID, TARGET]
assert len(submission) == len(sub)
assert (submission[ID].values == sub[ID].values).all(), "ids must match sample_submission in ORDER"
assert submission[TARGET].notna().all() and submission[TARGET].between(0, 1).all()
print(f"submission validated: {len(submission):,} rows, "
      f"mean {submission[TARGET].mean():.5f} (train base rate 0.70942)")

In [ ]:
RUN_METRICS = {
    "run_tag":            CFG["run_tag"],
    "final_oof_auc":      round(float(oof_auc), 6),
    f"{LEARNER}_oof_auc": round(float(oof_auc), 6),
    "fold_aucs":          [round(float(a), 6) for a in fold_aucs],
    "fold_auc_mean":      round(float(np.mean(fold_aucs)), 6),
    "fold_auc_std":       round(float(np.std(fold_aucs)), 6),
    "best_iters":         best_iters,
    # Count the columns actually FED TO THE MODEL, not len(FEATURES): the __te columns
    # are created inside the fold loop, so len(FEATURES) under-reports by len(TE_COLS)
    # and the log would claim a 12-feature model that really saw 21.
    "n_features":         int(Xtr.shape[1]),
    "engineered":         added,
    "te_cols":            TE_COLS,
    "model_seed":         CFG["model_seed"],
    "n_folds":            N_FOLDS,
    "cv_seed":            SEED,
    "cfg":                CFG,
    "notebook_runtime_sec": round(time.time() - T_START, 1),
}
print("RUN_METRICS_JSON:" + json.dumps(RUN_METRICS))